In [1]:
# render.py
import os, json, numpy as np
from PIL import Image, ImageDraw, ImageFont
from fontTools.ttLib import TTFont
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

TTF_DIR = './fonts/ttf'
OUT = './fonts/emb'
WORKERS = 32
os.makedirs(OUT, exist_ok=True)

DIGITS = [chr(c) for c in range(0x30, 0x3A)]
LATIN  = [chr(c) for c in range(0x41, 0x5B)] + [chr(c) for c in range(0x61, 0x7B)]
JAMO   = [chr(c) for c in range(0x3131, 0x318F) if chr(c).strip()]

# KS X 1001 한글 음절 2350자 (EUC-KR 0xB0A1–0xC8FE)
SYLL = []
for hi in range(0xB0, 0xC9):
    for lo in range(0xA1, 0xFF):
        try:
            ch = bytes([hi, lo]).decode('euc-kr')
        except UnicodeDecodeError:
            continue
        if '가' <= ch <= '힣':
            SYLL.append(ch)

assert len(DIGITS) == 10
assert len(LATIN) == 52
assert len(JAMO) == 94, len(JAMO)
assert len(SYLL) == 2350, len(SYLL)

GROUPS = [('digit', DIGITS), ('latin', LATIN), ('jamo', JAMO), ('syll', SYLL)]
ALL_CHARS = [c for _, g in GROUPS for c in g]
CHAR_INDEX = {c: i for i, c in enumerate(ALL_CHARS)}
N_CHARS = len(ALL_CHARS)  # 2506

IMG = 224
FONT_PX = 160
MM_PATH = f'{OUT}/glyphs.u8'

def render_glyph(pil_font, ch):
    img = Image.new('L', (IMG, IMG), 255)
    d = ImageDraw.Draw(img)
    l, t, r, b = d.textbbox((0, 0), ch, font=pil_font)
    w, h = r - l, b - t
    x = (IMG - w) / 2 - l
    y = (IMG - h) / 2 - t
    d.text((x, y), ch, font=pil_font, fill=0)
    return np.array(img, dtype=np.uint8)

def process_font(args):
    fi, fn, n_files = args
    font_path = os.path.join(TTF_DIR, fn)
    try:
        cmap = TTFont(font_path).getBestCmap()
        pil_font = ImageFont.truetype(font_path, FONT_PX)
    except Exception as e:
        return fi, None, f'open_fail: {e}'

    has_mask = np.array([ord(c) in cmap for c in ALL_CHARS], dtype=bool)
    arr = np.full((N_CHARS, IMG, IMG), 255, dtype=np.uint8)
    for i, c in enumerate(ALL_CHARS):
        if has_mask[i]:
            try:
                arr[i] = render_glyph(pil_font, c)
            except Exception:
                has_mask[i] = False

    mm = np.memmap(MM_PATH, mode='r+', dtype=np.uint8,
                   shape=(n_files, N_CHARS, IMG, IMG))
    mm[fi] = arr
    mm.flush()
    del mm
    return fi, has_mask, None

if __name__ == '__main__':
    files = sorted(f for f in os.listdir(TTF_DIR) if f.endswith('.ttf'))
    n = len(files)
    masks = np.zeros((n, N_CHARS), dtype=bool)

    # 메모리맵 파일 미리 생성 (workers가 r+로 연다)
    mm = np.memmap(MM_PATH, mode='w+', dtype=np.uint8,
                   shape=(n, N_CHARS, IMG, IMG))
    del mm

    tasks = [(i, fn, n) for i, fn in enumerate(files)]
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(process_font, t): t for t in tasks}
        for fut in tqdm(as_completed(futs), total=n):
            fi, m, err = fut.result()
            if err:
                print('fail', files[fi], err)
                continue
            masks[fi] = m

    np.save(f'{OUT}/masks.npy', masks)
    with open(f'{OUT}/files.json', 'w') as f:
        json.dump(files, f, ensure_ascii=False)

100%|██████████| 1237/1237 [08:10<00:00,  2.52it/s]


In [2]:
# embed.py
import json, numpy as np, torch
from transformers import AutoImageProcessor, AutoModel
from tqdm import tqdm

OUT = './fonts/emb'
IMG = 224
N_CHARS = 2506
GROUP_SLICES = {  # render.py와 순서 동일해야 함
    'digit': (0, 10),
    'latin': (10, 62),
    'jamo':  (62, 156),
    'syll':  (156, 2506),
}

device = 'cuda'
MODEL_ID = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
proc = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

files = json.load(open(f'{OUT}/files.json'))
masks = np.load(f'{OUT}/masks.npy')        # (F, N)
glyphs = np.memmap(f'{OUT}/glyphs.u8', mode='r', dtype=np.uint8,
                   shape=(len(files), N_CHARS, IMG, IMG))

D = model.config.hidden_size
emb = np.zeros((len(files), N_CHARS, D), dtype=np.float32)

BATCH = 256

@torch.inference_mode()
def encode(batch_u8):
    # batch_u8: (B, 224, 224) uint8 grayscale → RGB
    x = torch.from_numpy(batch_u8).to(device)
    x = x.unsqueeze(1).repeat(1, 3, 1, 1).float() / 255.0
    # DINOv3 normalize
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)
    x = (x - mean) / std
    out = model(pixel_values=x).last_hidden_state[:, 0]  # CLS
    return out.cpu().numpy()

for fi in tqdm(range(len(files))):
    idx = np.where(masks[fi])[0]
    if len(idx) == 0:
        continue
    for s in range(0, len(idx), BATCH):
        sub = idx[s:s+BATCH]
        emb[fi, sub] = encode(glyphs[fi, sub])

np.save(f'{OUT}/emb_raw.npy', emb)

# --- 글자별 평균 빼서 정체성 제거 ---
# 마스크된 위치만 평균에 포함
mask_f = masks.astype(np.float32)            # (F, N)
denom = mask_f.sum(0).clip(min=1)[:, None]   # (N, 1)
char_mean = (emb * mask_f[:, :, None]).sum(0) / denom  # (N, D)
emb_resid = emb - char_mean[None]            # (F, N, D)
emb_resid *= mask_f[:, :, None]              # 없는 글자는 0

# --- 그룹 concat ---
parts, present_bits = [], []
for name, (a, b) in GROUP_SLICES.items():
    g_mask = mask_f[:, a:b]                  # (F, n_g)
    g_emb  = emb_resid[:, a:b]               # (F, n_g, D)
    cnt = g_mask.sum(1, keepdims=True).clip(min=1)
    g_mean = g_emb.sum(1) / cnt              # (F, D)
    parts.append(g_mean)
    present_bits.append((g_mask.sum(1) > 0).astype(np.float32)[:, None])

font_vec = np.concatenate(parts + present_bits, axis=1)  # (F, 4D + 4)
# L2 정규화
font_vec /= (np.linalg.norm(font_vec, axis=1, keepdims=True) + 1e-9)
np.save(f'{OUT}/font_vec.npy', font_vec)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

100%|██████████| 1237/1237 [54:11<00:00,  2.63s/it]


In [3]:
import numpy as np
from sklearn.manifold import TSNE

X = np.load('./fonts/emb/font_vec.npy').astype(np.float32)
Y = TSNE(n_components=2, perplexity=30, learning_rate=200,
         init='pca', n_jobs=-1, random_state=0).fit_transform(X)
np.save('./fonts/emb/tsne.npy', Y.astype(np.float32))

In [7]:
# app.py
import json, numpy as np, os, io, base64
import plotly.graph_objects as go
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

TTF_DIR = './fonts/ttf'
OUT_HTML = './fonts/emb/tsne.html'
files = json.load(open('./fonts/emb/files.json'))
Y = np.load('./fonts/emb/tsne.npy')

SAMPLE = '가낡더떴끼릭맺료콤으숲황\nHaReSg125680'
FONT_PX = 56
PAD = 10

def sample_b64(fn):
    try:
        f = ImageFont.truetype(os.path.join(TTF_DIR, fn), FONT_PX)
        dummy = ImageDraw.Draw(Image.new('L', (10, 10)))
        l, t, r, b = dummy.multiline_textbbox((0, 0), SAMPLE, font=f, spacing=4)
        w = (r - l) + 2 * PAD
        h = (b - t) + 2 * PAD
    except Exception:
        f = None
        w, h = 200, FONT_PX * 2 + 2 * PAD
    img = Image.new('L', (w, h), 255)
    d = ImageDraw.Draw(img)
    if f is not None:
        d.multiline_text((PAD - l, PAD - t), SAMPLE, font=f, fill=0, spacing=4)
    else:
        d.text((PAD, PAD), '?', fill=0)
    buf = io.BytesIO(); img.save(buf, 'PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
    
print('rendering samples...')
imgs = [sample_b64(fn) for fn in tqdm(files)]

fig = go.Figure(go.Scatter(
    x=Y[:, 0], y=Y[:, 1], mode='markers',
    marker=dict(size=6, opacity=0.75, color='steelblue'),
    customdata=list(zip(files, imgs)),
    hovertemplate='%{customdata[0]}<extra></extra>',
))
fig.update_layout(height=900, dragmode='pan',
                  margin=dict(l=0, r=0, t=0, b=0),
                  plot_bgcolor='white')

plot_div = fig.to_html(full_html=False, include_plotlyjs='cdn',
                       div_id='plot')

html = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>font tsne</title>
<style>
  body {{ margin:0; font-family:sans-serif; }}
  #wrap {{ position:relative; }}
  #preview {{
    position:fixed; pointer-events:none;
    background:white; border:1px solid #888; padding:4px;
    box-shadow:0 2px 8px rgba(0,0,0,.2);
    display:none; z-index:1000;
  }}
  #preview img {{ display:block; }}
  #preview div {{ font-size:12px; margin-top:2px; }}
</style>
</head><body>
<div id="wrap">{plot_div}</div>
<div id="preview"><img id="pimg"><div id="ptxt"></div></div>
<script>
const plot = document.getElementById('plot');
const prev = document.getElementById('preview');
const pimg = document.getElementById('pimg');
const ptxt = document.getElementById('ptxt');

plot.on('plotly_hover', e => {{
  const p = e.points[0];
  const [name, src] = p.customdata;
  pimg.src = src;
  ptxt.textContent = name;
  prev.style.display = 'block';
}});
plot.on('plotly_unhover', () => {{ prev.style.display = 'none'; }});

document.addEventListener('mousemove', e => {{
  prev.style.left = (e.clientX + 16) + 'px';
  prev.style.top  = (e.clientY + 16) + 'px';
}});
</script>
</body></html>
"""

with open(OUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)
print('saved', OUT_HTML)

rendering samples...


100%|██████████| 1237/1237 [00:04<00:00, 262.42it/s]


saved ./fonts/emb/tsne.html


### 여기부터는 2글자 랜덤크롭

In [1]:
# render.py
import os, json, random, io, numpy as np
from PIL import Image, ImageDraw, ImageFont
from fontTools.ttLib import TTFont
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

TTF_DIR = './fonts/ttf'
OUT = './fonts/emb'
PNG_DIR = f'{OUT}/png'
WORKERS = 32
N_PER_FONT = 8192
GLYPH_H = 200
IMG = 224
PAD = 40
SEED = 0
os.makedirs(PNG_DIR, exist_ok=True)

DIGITS = [chr(c) for c in range(0x30, 0x3A)]
LATIN  = [chr(c) for c in range(0x41, 0x5B)] + [chr(c) for c in range(0x61, 0x7B)]
JAMO   = [chr(c) for c in range(0x3131, 0x318F) if chr(c).strip()]
SYLL = []
for hi in range(0xB0, 0xC9):
    for lo in range(0xA1, 0xFF):
        try:
            ch = bytes([hi, lo]).decode('euc-kr')
        except UnicodeDecodeError:
            continue
        if '가' <= ch <= '힣':
            SYLL.append(ch)
POOL = DIGITS + LATIN + JAMO + SYLL

def cmap_set(font_path):
    return set(TTFont(font_path).getBestCmap().keys())

def fit_font_size(font_path, sample_char):
    lo, hi, best = 8, 800, 100
    for _ in range(20):
        mid = (lo + hi) // 2
        try:
            f = ImageFont.truetype(font_path, mid)
            l, t, r, b = f.getbbox(sample_char)
            h = b - t
        except Exception:
            return 100
        if h < GLYPH_H:
            best = mid; lo = mid + 1
        else:
            hi = mid - 1
    return max(8, best)

def render_two(font_path, chars, px):
    f = ImageFont.truetype(font_path, px)
    text = ''.join(chars)
    dummy = Image.new('L', (10, 10), 255)
    l, t, r, b = ImageDraw.Draw(dummy).textbbox((0, 0), text, font=f)
    w = max(IMG + 1, (r - l) + 2 * PAD)
    img = Image.new('L', (w, IMG), 255)
    d = ImageDraw.Draw(img)
    d.text((PAD - l, (IMG - (b - t)) // 2 - t), text, font=f, fill=0)
    return np.array(img, dtype=np.uint8)

def random_crop(arr, rng):
    h, w = arr.shape
    x = rng.randint(0, w - IMG)
    return arr[:, x:x+IMG]

def encode_png(arr):
    img = Image.fromarray(arr, 'L')
    buf = io.BytesIO()
    img.save(buf, 'PNG', optimize=False, compress_level=6)
    return buf.getvalue()

def process_font(args):
    fi, fn = args
    path = os.path.join(TTF_DIR, fn)
    out_path = os.path.join(PNG_DIR, f'{fi:06d}.npz')
    if os.path.exists(out_path):
        return fi, True, 'skip'
    rng = random.Random(SEED + fi)
    try:
        cs = cmap_set(path)
        usable = [c for c in POOL if ord(c) in cs]
        if len(usable) < 2:
            return fi, False, 'too_few_chars'
        sample = 'M' if 'M' in usable else usable[0]
        px = fit_font_size(path, sample)
    except Exception as e:
        return fi, False, f'open_fail: {e}'

    blobs = []
    for k in range(N_PER_FONT):
        try:
            pair = rng.sample(usable, 2)
            full = render_two(path, pair, px)
            crop = random_crop(full, rng)
            blobs.append(encode_png(crop))
        except Exception:
            blobs.append(encode_png(np.full((IMG, IMG), 255, np.uint8)))

    # 가변 길이 bytes를 npz로: 단일 byte array + offsets
    sizes = np.array([len(b) for b in blobs], dtype=np.int64)
    offsets = np.concatenate([[0], np.cumsum(sizes)])
    data = np.frombuffer(b''.join(blobs), dtype=np.uint8)
    np.savez(out_path, data=data, offsets=offsets)
    return fi, True, None

if __name__ == '__main__':
    files = sorted(f for f in os.listdir(TTF_DIR) if f.endswith('.ttf'))
    n = len(files)
    valid = np.zeros(n, dtype=bool)

    tasks = list(enumerate(files))
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(process_font, t): t for t in tasks}
        for fut in tqdm(as_completed(futs), total=n):
            fi, ok, err = fut.result()
            if not ok:
                print('skip', files[fi], err); continue
            valid[fi] = True

    np.save(f'{OUT}/valid.npy', valid)
    with open(f'{OUT}/files.json', 'w') as f:
        json.dump(files, f, ensure_ascii=False)
    with open(f'{OUT}/shape.json', 'w') as f:
        json.dump({'F': n, 'N': N_PER_FONT, 'H': IMG, 'W': IMG}, f)
    print('valid', int(valid.sum()), '/', n)

100%|██████████| 1237/1237 [16:28<00:00,  1.25it/s]

valid 1237 / 1237


In [ ]:
# train.py
import json, re, io, os, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from transformers import AutoModel
from PIL import Image
from tqdm import tqdm

OUT = './fonts/emb'
PNG_DIR = f'{OUT}/png'
CKPT_DIR = f'{OUT}/ckpt'
CKPT_BEST = f'{OUT}/clf.pt'
MODEL_ID = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
device = 'cuda'
EPOCHS = 15
BATCH = 2048
LR = 1e-3
VAL_FRAC = 0.02
NUM_WORKERS = 32
PREFETCH = 4
ALPHA = 0.7
SHIFT = 16
os.makedirs(CKPT_DIR, exist_ok=True)

shape = json.load(open(f'{OUT}/shape.json'))
F_all, N, H, W = shape['F'], shape['N'], shape['H'], shape['W']
files = json.load(open(f'{OUT}/files.json'))
valid = np.load(f'{OUT}/valid.npy')

keep_f = np.where(valid)[0]
F_cls = len(keep_f)
print('classes', F_cls, 'samples/class', N)

def family_of(fn):
    stem = re.sub(r'\.(ttf|otf)$', '', fn, flags=re.I)
    stem = re.sub(r'-VF\d+-\d+(-\d+)?$', '', stem)
    stem = re.sub(r'-\d+$', '', stem)
    return stem

fams = [family_of(files[fi]) for fi in keep_f]
fam2members = {}
for ci, f in enumerate(fams):
    fam2members.setdefault(f, []).append(ci)
sibling_lists = [np.array(fam2members[f], dtype=np.int64) for f in fams]

soft_lut = torch.zeros(F_cls, F_cls, dtype=torch.float32)
for ci in range(F_cls):
    sibs = sibling_lists[ci]
    if len(sibs) == 1:
        soft_lut[ci, ci] = 1.0
    else:
        soft_lut[ci, sibs] = (1.0 - ALPHA) / (len(sibs) - 1)
        soft_lut[ci, ci] = ALPHA
soft_lut = soft_lut.to(device)
sibling_set = [set(s.tolist()) for s in sibling_lists]

print('loading PNG blobs to RAM...')
blob_data = [None] * F_cls
blob_off  = [None] * F_cls
total_bytes = 0
for ci, fi in enumerate(tqdm(keep_f)):
    z = np.load(os.path.join(PNG_DIR, f'{fi:06d}.npz'))
    blob_data[ci] = z['data']
    blob_off[ci]  = z['offsets']
    total_bytes += blob_data[ci].nbytes
print(f'loaded {total_bytes/1e9:.1f} GB')

rng = np.random.RandomState(0)
perm = rng.permutation(N)
n_val = max(1, int(N * VAL_FRAC))
val_slots   = sorted(perm[:n_val].tolist())
train_slots = sorted(perm[n_val:].tolist())

class FontDS(Dataset):
    def __init__(self, slots, augment):
        self.slots = np.array(slots, dtype=np.int64)
        self.n_slots = len(self.slots)
        self.augment = augment
    def __len__(self):
        return F_cls * self.n_slots
    def __getitem__(self, i):
        ci = i // self.n_slots
        si = i %  self.n_slots
        k = int(self.slots[si])
        offs = blob_off[ci]
        png = blob_data[ci][offs[k]:offs[k+1]]
        x = np.array(Image.open(io.BytesIO(png.tobytes())), dtype=np.uint8)
        if self.augment:
            s = np.random.randint(-SHIFT, SHIFT + 1)
            if s != 0:
                x = np.roll(x, s, axis=1)
                if s > 0: x[:, :s] = 255
                else:     x[:, s:] = 255
        return x, ci

def collate(batch):
    x = torch.from_numpy(np.stack([b[0] for b in batch]))
    y = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return x, y

train_dl = DataLoader(FontDS(train_slots, augment=True), batch_size=BATCH, shuffle=True,
                      num_workers=NUM_WORKERS, collate_fn=collate,
                      drop_last=True, persistent_workers=True,
                      pin_memory=True, prefetch_factor=PREFETCH)
val_dl   = DataLoader(FontDS(val_slots, augment=False), batch_size=BATCH, shuffle=False,
                      num_workers=NUM_WORKERS, collate_fn=collate,
                      persistent_workers=True, pin_memory=True,
                      prefetch_factor=PREFETCH)

backbone = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
for p in backbone.parameters(): p.requires_grad = False
D = backbone.config.hidden_size

head = nn.Linear(D, F_cls).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

def feats(x_u8):
    x = x_u8.to(device, non_blocking=True).float() / 255.0
    x = x.unsqueeze(1).repeat(1,3,1,1)
    x = (x - mean) / std
    with torch.no_grad(), autocast('cuda', dtype=torch.bfloat16):
        out = backbone(pixel_values=x).last_hidden_state[:, 0]
    return out.float()

def soft_ce(logits, target_soft):
    logp = F.log_softmax(logits, dim=-1)
    return -(target_soft * logp).sum(-1).mean()

best = 0.0
for ep in range(EPOCHS):
    head.train()
    pbar = tqdm(train_dl, desc=f'ep{ep}')
    tot, c1, cf, loss_sum = 0, 0, 0, 0.0
    for x, y in pbar:
        f = feats(x)
        logits = head(f)
        y_dev = y.to(device, non_blocking=True)
        soft = soft_lut[y_dev]
        loss = soft_ce(logits, soft)
        opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item() * y.size(0)

        pred = logits.argmax(1).cpu()
        c1 += (pred == y).sum().item()
        for b in range(y.size(0)):
            if pred[b].item() in sibling_set[int(y[b])]:
                cf += 1
        tot += y.size(0)
        pbar.set_postfix(loss=f'{loss_sum/tot:.3f}',
                         acc=f'{c1/tot:.3f}',
                         fam=f'{cf/tot:.3f}')
    sched.step()

    head.eval()
    vt, v1, vf = 0, 0, 0
    with torch.no_grad():
        for x, y in val_dl:
            logits = head(feats(x))
            pred = logits.argmax(1).cpu()
            v1 += (pred == y).sum().item()
            for b in range(y.size(0)):
                if pred[b].item() in sibling_set[int(y[b])]:
                    vf += 1
            vt += y.size(0)
    vacc = v1 / max(vt, 1)
    vfam = vf / max(vt, 1)
    print(f'ep{ep} val acc {vacc:.4f} fam {vfam:.4f}')

    ckpt = {'head': head.state_dict(), 'keep_f': keep_f, 'files': files,
            'epoch': ep, 'val_acc': vacc, 'val_fam': vfam}
    ep_path = f'{CKPT_DIR}/ep{ep:02d}_acc{vacc:.4f}.pt'
    torch.save(ckpt, ep_path)
    print('saved', ep_path)
    if vacc > best:
        best = vacc
        torch.save(ckpt, CKPT_BEST)
        print('saved best', CKPT_BEST)


classes 1237 samples/class 8192
loading PNG blobs to RAM...


100%|██████████| 1237/1237 [00:12<00:00, 96.72it/s] 


loaded 24.4 GB


Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

ep0:  32%|███▏      | 1562/4849 [21:29<43:21,  1.26it/s, acc=0.429, fam=0.479, loss=3.264] 

In [ ]:
# tsne.py
import json, numpy as np, torch
from transformers import AutoModel
from tqdm import tqdm
from tsnecuda import TSNE

OUT = './fonts/emb'
MODEL_ID = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
device = 'cuda'
BATCH = 2048
N_FOR_FONT_VEC = 256       # 폰트 평균 낼 때 사용할 슬롯 수
N_FOR_CROP_TSNE = 16       # crop-level t-SNE에 폰트당 몇 점 쓸지

shape = json.load(open(f'{OUT}/shape.json'))
F_all, N, H, W = shape['F'], shape['N'], shape['H'], shape['W']
files = json.load(open(f'{OUT}/files.json'))
valid = np.load(f'{OUT}/valid.npy')
mm = np.memmap(f'{OUT}/crops.u8', mode='r', dtype=np.uint8,
               shape=(F_all, N, H, W))

backbone = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
D = backbone.config.hidden_size
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

@torch.inference_mode()
def encode(batch_u8):
    x = torch.from_numpy(batch_u8).to(device).float() / 255.0
    x = x.unsqueeze(1).repeat(1,3,1,1)
    x = (x - mean) / std
    return backbone(pixel_values=x).last_hidden_state[:, 0].cpu().numpy()

keep_f = np.where(valid)[0]
slots_avg  = np.arange(N_FOR_FONT_VEC)
slots_crop = np.arange(N_FOR_CROP_TSNE)

# 폰트 벡터: N_FOR_FONT_VEC개 평균
font_vec = np.zeros((len(keep_f), D), dtype=np.float32)
for i, fi in enumerate(tqdm(keep_f, desc='font_vec')):
    chunk = np.asarray(mm[fi, :N_FOR_FONT_VEC])  # (n,224,224)
    accs = []
    for s in range(0, len(chunk), BATCH):
        accs.append(encode(chunk[s:s+BATCH]))
    font_vec[i] = np.concatenate(accs).mean(0)
font_vec /= (np.linalg.norm(font_vec, axis=1, keepdims=True) + 1e-9)
np.save(f'{OUT}/font_vec.npy', font_vec)

Yf = TSNE(n_components=2, perplexity=30, learning_rate=200).fit_transform(font_vec.astype(np.float32))
np.save(f'{OUT}/tsne_font.npy', Yf)

# crop-level: 폰트당 N_FOR_CROP_TSNE점만
all_emb, all_lab = [], []
for i, fi in enumerate(tqdm(keep_f, desc='crops')):
    chunk = np.asarray(mm[fi, :N_FOR_CROP_TSNE])
    e = encode(chunk)
    all_emb.append(e)
    all_lab.append(np.full(len(e), i, dtype=np.int64))
all_emb = np.concatenate(all_emb)
all_lab = np.concatenate(all_lab)
all_emb /= (np.linalg.norm(all_emb, axis=1, keepdims=True) + 1e-9)
Yc = TSNE(n_components=2, perplexity=50, learning_rate=200).fit_transform(all_emb.astype(np.float32))
np.save(f'{OUT}/tsne_crops.npy', Yc)
np.save(f'{OUT}/tsne_crops_labels.npy', all_lab)
print('done', Yf.shape, Yc.shape)

In [ ]:
# app.py
import json, io, base64, numpy as np
import gradio as gr
import plotly.graph_objects as go
from PIL import Image

OUT = './fonts/emb'
files = json.load(open(f'{OUT}/files.json'))
valid = np.load(f'{OUT}/valid.npy')
keep_f = np.where(valid)[0]
Yf = np.load(f'{OUT}/tsne_font.npy')
Yc = np.load(f'{OUT}/tsne_crops.npy')
Lc = np.load(f'{OUT}/tsne_crops_labels.npy')
crops = np.load(f'{OUT}/crops.npy')
N = crops.shape[1]

def crop_b64(fi, k=0):
    img = Image.fromarray(crops[fi, k], 'L')
    buf = io.BytesIO(); img.save(buf, 'PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

def fig_font():
    names = [files[i] for i in keep_f]
    return go.Figure(go.Scatter(
        x=Yf[:,0], y=Yf[:,1], mode='markers',
        marker=dict(size=6, opacity=0.75),
        text=names, hoverinfo='text',
    )).update_layout(height=720, dragmode='pan',
                     title='font-level (1 point per ttf)')

def fig_crops():
    return go.Figure(go.Scatter(
        x=Yc[:,0], y=Yc[:,1], mode='markers',
        marker=dict(size=3, opacity=0.5, color=Lc % 20, colorscale='hsv'),
        text=[files[i] for i in Lc], hoverinfo='text',
    )).update_layout(height=720, dragmode='pan',
                     title=f'crop-level ({N} per ttf)')

def on_select_font(evt: gr.SelectData):
    fi = int(keep_f[evt.index])
    html = '<div style="display:flex;gap:4px;flex-wrap:wrap">'
    for k in range(min(8, N)):
        html += f'<img src="{crop_b64(fi,k)}" style="width:112px;height:112px;border:1px solid #ccc">'
    html += '</div>'
    return files[fi], html

def on_select_crops(evt: gr.SelectData):
    idx = evt.index
    fi = int(Lc[idx])
    k = idx % N
    return files[fi], f'<img src="{crop_b64(fi,k)}" style="width:224px;height:224px;border:1px solid #ccc">'

with gr.Blocks() as demo:
    with gr.Tab('font-level'):
        with gr.Row():
            p1 = gr.Plot(fig_font(), scale=3)
            with gr.Column(scale=1):
                n1 = gr.Textbox(label='font'); h1 = gr.HTML()
        p1.select(on_select_font, None, [n1, h1])
    with gr.Tab('crop-level'):
        with gr.Row():
            p2 = gr.Plot(fig_crops(), scale=3)
            with gr.Column(scale=1):
                n2 = gr.Textbox(label='font'); h2 = gr.HTML()
        p2.select(on_select_crops, None, [n2, h2])

demo.launch()

In [ ]:
# app.py
import json, io, base64, numpy as np
import gradio as gr
import plotly.graph_objects as go
from PIL import Image

OUT = './fonts/emb'
shape = json.load(open(f'{OUT}/shape.json'))
F_all, N, H, W = shape['F'], shape['N'], shape['H'], shape['W']
files = json.load(open(f'{OUT}/files.json'))
valid = np.load(f'{OUT}/valid.npy')
keep_f = np.where(valid)[0]
Yf = np.load(f'{OUT}/tsne_font.npy')
Yc = np.load(f'{OUT}/tsne_crops.npy')
Lc = np.load(f'{OUT}/tsne_crops_labels.npy')

mm = np.memmap(f'{OUT}/crops.u8', mode='r', dtype=np.uint8,
               shape=(F_all, N, H, W))
N_PER_FONT_IN_CROP_TSNE = (Lc == 0).sum()  # crop-level에서 폰트당 점 수

def crop_b64(fi, k=0):
    img = Image.fromarray(np.asarray(mm[fi, k]), 'L')
    buf = io.BytesIO(); img.save(buf, 'PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

def fig_font():
    names = [files[i] for i in keep_f]
    return go.Figure(go.Scatter(
        x=Yf[:,0], y=Yf[:,1], mode='markers',
        marker=dict(size=6, opacity=0.75),
        text=names, hoverinfo='text',
    )).update_layout(height=720, dragmode='pan',
                     title='font-level (1 point per ttf)')

def fig_crops():
    return go.Figure(go.Scatter(
        x=Yc[:,0], y=Yc[:,1], mode='markers',
        marker=dict(size=3, opacity=0.5, color=Lc % 20, colorscale='hsv'),
        text=[files[keep_f[i]] for i in Lc], hoverinfo='text',
    )).update_layout(height=720, dragmode='pan',
                     title=f'crop-level ({int(N_PER_FONT_IN_CROP_TSNE)} per ttf)')

def on_select_font(evt: gr.SelectData):
    fi = int(keep_f[evt.index])
    html = '<div style="display:flex;gap:4px;flex-wrap:wrap">'
    for k in range(8):
        html += f'<img src="{crop_b64(fi,k)}" style="width:112px;height:112px;border:1px solid #ccc">'
    html += '</div>'
    return files[fi], html

def on_select_crops(evt: gr.SelectData):
    idx = evt.index
    ci = int(Lc[idx])
    fi = int(keep_f[ci])
    k = idx % int(N_PER_FONT_IN_CROP_TSNE)
    return files[fi], f'<img src="{crop_b64(fi,k)}" style="width:224px;height:224px;border:1px solid #ccc">'

with gr.Blocks() as demo:
    with gr.Tab('font-level'):
        with gr.Row():
            p1 = gr.Plot(fig_font(), scale=3)
            with gr.Column(scale=1):
                n1 = gr.Textbox(label='font'); h1 = gr.HTML()
        p1.select(on_select_font, None, [n1, h1])
    with gr.Tab('crop-level'):
        with gr.Row():
            p2 = gr.Plot(fig_crops(), scale=3)
            with gr.Column(scale=1):
                n2 = gr.Textbox(label='font'); h2 = gr.HTML()
        p2.select(on_select_crops, None, [n2, h2])

demo.launch()